# Week 3 Assignment
- **Assignment Description** The overall goal of this assignment is to use Conditional Random Fields (CRF) to extract named entities.

- You will train a CRF, and then iteratively improve it.

- You will use a validation set to work on your model, then ultimately it will be tested on an unseen test set.

## Data File Description

The dataset is about psychiatric treatment adverse reactions. The dataset covers patients' expression of effectiveness and adverse drug events associated with four psychiatric medications.

For this assignment, you will need files list below.
```
/review_data
    REVIEW_LABELSEQ.txt
    REVIEW_TEXT.txt
    TEST_REVIEW_TEXT.txt
```

1. `REVIEW_TEXT.txt`
> The training file contains 4,744 sentences coming from 711 reviews. You can find that all patients' reviews are split by sentence, and the file has two tab-separated columns. One of the columns is the ID for each sentence, which is labeled as `<Medication>.<Post_number>.<Sentence_number>`. The other column is the split sentence itself. 
  > Here is an example row of this dataset.

  >  `drug1.Post1.Sentence8<tab>Tobacco cravings were rampant .`
  


2. `REVIEW_LABELSEQ.txt`
> You can find there are two columns, ID and TAGSEQ. ID is the same unique `<Medication>.<Post_number>.<Sentence_number>` strings as previous file. TAGSEQ is a sequence of sapce-separated named-entity tags that are either `O` or `B-AE`, `I-AE`, `B-SSI`, `I-SSI`. These tags are explained as below.
  - `AE`: adverse events (entity of interest)
  - `SSI`: signs, symptoms, and indications (entity of interest)
  - `B-`: beginning of a tagged named entity
  - `I-`: inside a tagged named entity
  - `O` : outside of any tagged named entity

  > Here is an example row of this file.

  >  `8<tab>B-AE I-AE I-AE I-AE O `


3. `TEST_REVIEW_TEXT`
> The test file has 1,259 sentences from 180 reviews. The structure is the same as `REVIEW_TEXT.txt` file.




## Code Template
Please modify code templates as much as you want. If you find using a function would be more useful, please use it!

### Step 1: Read File

In [1]:
def read_file(f):
    """This function is used to read files that are tab-separated. 
    The function will split each row into two parts: ID and data.
    Data is a list of either sentence or tag sequence that is split into a list by space. 
    """
    data = open(f,'r').readlines()[1:]
    row_id = [i.split('\t')[0].strip() for i in data]
    data = [i.split('\t')[1].strip().split(' ') for i in data]
    return row_id,data

In [2]:
row_id_text, texts = read_file('./data/REVIEW_TEXT.txt')
row_id_tags, tags = read_file('./data/REVIEW_LABELSEQ.txt')

In [3]:
# notice that LABELSEQ and TEXT files have the same IDs in the same order
row_id_tags == row_id_text

True

In [4]:
texts[:100] # uncomment this row if you want to print out first 100 rows of TEXT file

[['Total', 'nightmare.'],
 ['This',
  'was',
  'arguably',
  'the',
  'worst',
  'period',
  'of',
  'time',
  'in',
  'my',
  'life.'],
 ['It',
  'made',
  'me',
  'so',
  'mentally',
  'ill',
  'I',
  'almost',
  'ended',
  'up',
  'in',
  'a',
  'psych',
  'ward.'],
 ['Stay', 'away', 'from', 'this', 'if', 'you', 'can!'],
 ['I',
  'lost',
  'weight',
  'considerably',
  'and',
  'shook',
  'all',
  'the',
  'time',
  '.'],
 ['I',
  'had',
  'terrible',
  'anxiety',
  'the',
  'whole',
  'time',
  ',',
  'the',
  'worst',
  'kind',
  'of',
  'anxiety',
  "I've",
  'ever',
  'experienced.'],
 ['Also', 'tardive', 'diskenesia', 'was', 'becoming', 'very', 'prevalent.'],
 ['Tobacco', 'cravings', 'were', 'rampant', '.'],
 ['I',
  'never',
  'felt',
  'at',
  'ease',
  'or',
  'comfortable',
  ',',
  'I',
  'was',
  'constantly',
  'on',
  'edge',
  'and',
  'wanted',
  'to',
  'cry',
  '.'],
 ['Please', 'be', 'careful', 'if', 'you', 'are', 'taking', 'this,', 'o'],
 ['Sleepy',
  ',',
  'cons

### Step 2: See how to Create Features

- We will use a function to turn words into features for our model to use  in training.

- By defining features as shown below, we can easily add more to the model later.

In [5]:
from stop_words import get_stop_words
import string # for punctuation

stop_words = get_stop_words('english')

def word2features(word, row_n):
    features = {
        'word.lower()': word.lower(),
        'stopword': 1 if word.lower() in stop_words else 0,  # 1 - is a stop word, 0 - not a stop word
        'punc': 1 if word.lower() in string.punctuation else 0,  # 1 - is a punctuation, 0 - not a punctuation
        'word.strip()': word.lower().strip(string.punctuation)
        # you will add other features that could help with the task later
    }
    return features

def text2features(text, row_n):
    # This function could change to incorporate sentence level information into word features
    return [word2features(i, row_n) for i in text]

### Step 3: Modify Data into X (inputs) and y (output)

In [6]:
import pandas as pd

df_texts = pd.DataFrame({'ID': row_id_text,
                        'texts': texts})

df_texts.head(5) # uncomment this row if you want to look the first 5 rows of the dataframe

,ID,texts
0,drug1.Post1.Sentence1,"[Total, nightmare.]"
1,drug1.Post1.Sentence2,"[This, was, arguably, the, worst, period, of, ..."
2,drug1.Post1.Sentence3,"[It, made, me, so, mentally, ill, I, almost, e..."
3,drug1.Post1.Sentence4,"[Stay, away, from, this, if, you, can!]"
4,drug1.Post1.Sentence5,"[I, lost, weight, considerably, and, shook, al..."


In [7]:
def createXy(text_list, text_feature_function):

    X = [text_feature_function(text_list[n], n) for n in range(len(texts))]
    y = tags
    return X, y
X, y = createXy(texts, text2features)

### Step 4: Split Data into Training and Validation set

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_validation, y_train, y_validation = train_test_split(X, y, test_size = 0.2)

In [9]:
# If you want to see what the examples look like, uncomment the lines below to
# view the first 10 examples in X_train and y_train

for X_, y_ in zip(X_train[:10], y_train[:10]):
    print(X_)
    print(y_)

[{'word.lower()': 'this', 'stopword': 1, 'punc': 0, 'word.strip()': 'this'}, {'word.lower()': 'drug', 'stopword': 0, 'punc': 0, 'word.strip()': 'drug'}, {'word.lower()': 'has', 'stopword': 1, 'punc': 0, 'word.strip()': 'has'}, {'word.lower()': 'worked', 'stopword': 0, 'punc': 0, 'word.strip()': 'worked'}, {'word.lower()': 'really', 'stopword': 0, 'punc': 0, 'word.strip()': 'really'}, {'word.lower()': 'well', 'stopword': 0, 'punc': 0, 'word.strip()': 'well'}, {'word.lower()': 'for', 'stopword': 1, 'punc': 0, 'word.strip()': 'for'}, {'word.lower()': 'my', 'stopword': 1, 'punc': 0, 'word.strip()': 'my'}, {'word.lower()': 'depression', 'stopword': 0, 'punc': 0, 'word.strip()': 'depression'}, {'word.lower()': ',', 'stopword': 0, 'punc': 1, 'word.strip()': ''}, {'word.lower()': 'and', 'stopword': 1, 'punc': 0, 'word.strip()': 'and'}, {'word.lower()': 'i', 'stopword': 1, 'punc': 0, 'word.strip()': 'i'}, {'word.lower()': 'feel', 'stopword': 0, 'punc': 0, 'word.strip()': 'feel'}, {'word.lower()

### Step 5: Build a default CRF model

In [10]:
from sklearn_crfsuite import CRF

crf = CRF(algorithm='lbfgs', max_iterations=100)

### Step 6: Train the CRF model

In [11]:
crf.fit(X_train, y_train)

CRF(algorithm='lbfgs', all_possible_states=None, all_possible_transitions=None,
    averaging=None, c=None, c1=None, c2=None, calibration_candidates=None,
    calibration_eta=None, calibration_max_trials=None, calibration_rate=None,
    calibration_samples=None, delta=None, epsilon=None, error_sensitive=None,
    gamma=None, keep_tempfiles=None, linesearch=None, max_iterations=100,
    max_linesearch=None, min_freq=None, model_filename=None, num_memories=None,
    pa_type=None, period=None, trainer_cls=None, variance=None, verbose=False)

### Step 7: Results - How the CRF model looks and performs

- (Sanity check) see which labels are in the model so you can report how many there are.

In [12]:
labels = list(crf.classes_)
labels

['O', 'B-SSI', 'B-AE', 'I-AE', 'I-SSI']

In [13]:
# Report the number of classes in the model by setting n to that number
n = len(labels)
print('Number of model classes:', n)

Number of model classes: 5


In [14]:
# test number of classes

In [15]:
# Report the number of classes in the model
n = len(labels)
print("Number of classes:", n)

# test number of classes
assert isinstance(n, int)
assert n == len(labels)
assert n == 5
print("Test passed! Number of classes:", n)

Number of classes: 5
Test passed! Number of classes: 5


In [16]:
from sklearn_crfsuite import metrics # for flat_classification_report and sequence_accuracy_score

def flat_classification_report(y_true, y_pred, labels=labels, digits=3):
    report = metrics.flat_classification_report(y_true, y_pred, labels=labels, digits=3)
    report += '\n'
    report += '{}{: >11}'.format('sequence acc', str(round(metrics.sequence_accuracy_score(y_true, y_pred), digits)))
    return report

In [17]:
from sklearn_crfsuite import metrics
y_pred = crf.predict(X_validation) # predictions with validation set

report = flat_classification_report(y_validation, y_pred, labels=labels, digits=3)
print(report)

              precision    recall  f1-score   support

           O      0.894     0.958     0.925     12009
       B-SSI      0.679     0.580     0.625       157
        B-AE      0.713     0.462     0.561       827
        I-AE      0.587     0.401     0.477      1506
       I-SSI      0.143     0.040     0.062       100

    accuracy                          0.862     14599
   macro avg      0.603     0.488     0.530     14599
weighted avg      0.845     0.862     0.849     14599

sequence acc      0.593


In the following cell, report the macro averaged f1-score from this iteration of traning.  

In [18]:
# Macro F1 across all reported labels, computed rather than guessed.
macro_avg_f1 = metrics.flat_f1_score(
    y_validation, y_pred, average='macro', labels=labels, zero_division=0
)
print('Baseline macro F1:', round(macro_avg_f1, 3))

Baseline macro F1: 0.53


In [19]:
# test macro_avg_f1


In [20]:
# test macro_avg_f1
assert isinstance(macro_avg_f1, (int, float)) or hasattr(macro_avg_f1, "item")
assert 0 <= macro_avg_f1 <= 1
print("Test passed! macro_avg_f1 =", round(macro_avg_f1, 3))

Test passed! macro_avg_f1 = 0.53


Report the support for B-AE and for B-SSI in the following cell

In [21]:
# Support = number of gold-standard tokens with the indicated label.
B_AE_support = sum(tag == 'B-AE' for sentence in y_validation for tag in sentence)
B_SSI_support = sum(tag == 'B-SSI' for sentence in y_validation for tag in sentence)
print('B-AE support:', B_AE_support, '| B-SSI support:', B_SSI_support)

B-AE support: 827 | B-SSI support: 157


### Step 8: Tune CRF model

- Now we will look at how to tune the model for better performance
- This tuning process can be a vital step in getting the most out of your models

In [22]:
import scipy.stats
from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_val_score, RandomizedSearchCV
from sklearn_crfsuite import scorers

In [23]:
def create_tuned_estimator():
    crf = CRF(
        algorithm='lbfgs',
        max_iterations=100,
        all_possible_transitions=True
    )
    params_space = {
        'c1': scipy.stats.expon(scale=0.5),
        'c2': scipy.stats.expon(scale=0.05),
    }

    # use the same metric for evaluation
    f1_scorer = make_scorer(metrics.flat_f1_score,
                            average='weighted', labels=labels)

    # Causes fit() to search for "best" model with these parameter settings
    rs = RandomizedSearchCV(crf, params_space,
                            cv=3,
                            verbose=1,
                            n_jobs=-1,
                            n_iter=50,
                            scoring=f1_scorer)
    return rs
    

In [ ]:
# Fit the model again!
rs = create_tuned_estimator()
rs.fit(X_train, y_train)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.


In [ ]:
crf_2 = rs.best_estimator_
y_pred_2 = crf_2.predict(X_validation)
print(flat_classification_report(
    y_validation, y_pred_2, labels=labels, digits=3
))

Let's focus on the macro averaged f1 score again.  Report that number from above as before in the following cell.

In [ ]:
macro_avg_f1_2 = metrics.flat_f1_score(
    y_validation, y_pred_2, average='macro', labels=labels, zero_division=0
)
print('Tuned macro F1:', round(macro_avg_f1_2, 3))

This above score should be higher than the original.  This is because we used an algorithm that searched more of the space to find a better model.

### Step 9: Add Your Own Features and retrain  (at least 2 features)

- Here we have the word2features function repeated, but called word2features_2.

- Add **at least** ***2*** more of your own features to the model by defining functions 

- You can define any function you want or use simple expressions like the features defined below!

- You are also allowed to remove features if you feel they are hurting performance.

In [ ]:

from stop_words import get_stop_words
import string
import nltk

stop_words = get_stop_words('english')

def word2features_2(word, row_n):
    features = {
        'word.lower()': word.lower(),
        'stopword': 1 if word.lower() in stop_words else 0,
        'punc': 1 if word in string.punctuation else 0,
        'word.strip()': word.lower().strip(string.punctuation),

        # Feature 1: Word shape
        'word.shape': ''.join(
            'X' if c.isupper() else
            'x' if c.islower() else
            'd' if c.isdigit() else c
            for c in word
        ),

        # Feature 2: Suffix
        'suffix-3': word.lower()[-3:],

        # Additional features
        'prefix-3': word.lower()[:3],
        'word.length': len(word),
        'word.istitle()': word.istitle(),
        'word.isupper()': word.isupper(),
        'word.isdigit()': word.isdigit(),
    }

    return features


def text2features_2(text, row_n):
    features = []

    for i, word in enumerate(text):
        word_features = word2features_2(word, row_n)

        # Previous word
        if i > 0:
            word_features['-1:word.lower()'] = text[i-1].lower()
        else:
            word_features['BOS'] = True

        # Next word
        if i < len(text) - 1:
            word_features['+1:word.lower()'] = text[i+1].lower()
        else:
            word_features['EOS'] = True

        features.append(word_features)

    return features


In [ ]:
# check your features

Of course, you need to apply the new features to the data, split into train and validation again, and train again:

In [ ]:
# Modify into X (inputs) and y (labels) again
# Check that the new feature generator produces the required extra features.
feature_example = text2features_2(['Severe', 'headache', 'today', '.'], 0)
assert len(feature_example) == 4
assert 'word.shape' in feature_example[1]
assert 'suffix-3' in feature_example[1]
assert '-1:word.lower()' in feature_example[1]
assert '+1:word.lower()' in feature_example[1]
print('Feature check passed:', feature_example[1])
X_2, y_2 = createXy(texts, text2features_2)
X_train_2, X_validation_2, y_train_2, y_validation_2 = train_test_split(X_2, y_2, test_size = 0.2)

# train a model again
rs_2 = create_tuned_estimator()
rs_2.fit(X_train_2, y_train_2)


In [ ]:
# Finally, make predictions once more and view results on validation
crf_3 = rs_2.best_estimator_
y_pred_3 = crf_3.predict(X_validation_2)
print(flat_classification_report(
    y_validation_2, y_pred_3, labels=labels, digits=3
))

### Step 10: Report Final performance

In [ ]:
macro_avg_f1_final = macro_avg_f1_final = metrics.flat_f1_score(
    y_validation_2, y_pred_3, average='macro', labels=labels, zero_division=0
)
print('Baseline macro F1:       ', round(macro_avg_f1, 3))
print('Tuned macro F1:          ', round(macro_avg_f1_2, 3))
print('Feature-enhanced macro F1:', round(macro_avg_f1_final, 3))
print('Gain versus baseline:    ', round(macro_avg_f1_final - macro_avg_f1, 3))

In [ ]:

# Test your improved model

from sklearn_crfsuite import metrics

# Verify that the improved model was trained
assert crf_3 is not None

# Verify predictions
assert len(y_pred_3) == len(y_validation_2)

# Calculate the final macro-averaged F1 score
macro_avg_f1_final = metrics.flat_f1_score(
    y_validation_2,
    y_pred_3,
    average='macro',
    labels=labels,
    zero_division=0
)

# Validate the result
assert 0 <= macro_avg_f1_final <= 1

# Compare model performance
print("Baseline F1:", round(macro_avg_f1, 3))
print("Tuned F1:", round(macro_avg_f1_2, 3))
print("Improved Model F1:", round(macro_avg_f1_final, 3))

if macro_avg_f1_final > macro_avg_f1:
    print("Improved model outperforms baseline!")
else:
    print("Additional feature tuning may be needed.")

print("All improved model tests passed!")


In [ ]:
# Test your improved model